In [ ]:
# Entregable 1 - Prediccion de Precios de Diamantes (CRISP-DM)
# Dataset: diamonds (seaborn)
# Tipo de problema: Regresion supervisada
# Variable objetivo: price (USD)

In [ ]:
# Librerias
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import skew
from sklearn.preprocessing import OrdinalEncoder

In [ ]:
# Tema global para graficos
sns.set_theme(style="whitegrid", palette="muted")

In [ ]:
# =============================================================
# 1. BUSINESS UNDERSTANDING
# =============================================================
# Problema: estimar el precio de un diamante a partir de sus
# caracteristicas fisicas y de calidad (4C: Carat, Cut, Color, Clarity)
# Justificacion: la tasacion manual es subjetiva, lenta y no escalable
# Metrica clave: R2 >= 0.85
# Metricas complementarias: MAE < $800 USD, RMSE

In [ ]:
# =============================================================
# 2. DATA UNDERSTANDING
# =============================================================

In [ ]:
# Carga del dataset
df = sns.load_dataset("diamonds")
print(f"Registros: {df.shape[0]}, Variables: {df.shape[1]}")

In [ ]:
# Primeros registros
df.head(10)

In [ ]:
# Estructura del dataset
df.info()

In [ ]:
# Resumen estadistico
df.describe()

In [ ]:
# Resumen estadistico de variables categoricas
df.describe(include="category")

In [ ]:
# Valores nulos por columna
print(df.isnull().sum())
print(f"\nTotal de nulos en el dataset: {df.isnull().sum().sum()}")

In [ ]:
# Mapa de valores faltantes
plt.figure(figsize=(10, 3))
sns.heatmap(df.isnull(), cbar=False, cmap="YlOrRd", yticklabels=False)
plt.title("Mapa de Valores Faltantes")
plt.tight_layout()
plt.show()
# No existen valores nulos explicitos en ninguna columna

In [ ]:
# Deteccion de valores anomalos: dimensiones fisicas en cero
# Un diamante no puede tener longitud, ancho o profundidad = 0
print("Registros con dimensiones = 0:")
print(f"  x = 0: {(df['x'] == 0).sum()}")
print(f"  y = 0: {(df['y'] == 0).sum()}")
print(f"  z = 0: {(df['z'] == 0).sum()}")
print(f"  Total afectados: {((df['x'] == 0) | (df['y'] == 0) | (df['z'] == 0)).sum()}")

In [ ]:
# Boxplots de variables numericas para detectar outliers
num_cols = df.select_dtypes(include=np.number).columns.tolist()
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    sns.boxplot(data=df, y=col, ax=axes[i])
    axes[i].set_title(col)
for j in range(len(num_cols), len(axes)):
    axes[j].set_visible(False)
plt.suptitle("Deteccion de Outliers por Variable Numerica", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Distribucion de la variable objetivo (price)
print(f"Asimetria (skewness) de price: {skew(df['price']):.4f}")
print(f"Media: ${df['price'].mean():.2f}")
print(f"Mediana: ${df['price'].median():.2f}")

plt.figure(figsize=(10, 5))
sns.histplot(df["price"], bins=50, kde=True)
plt.title("Distribucion del Precio de Diamantes")
plt.xlabel("Precio (USD)")
plt.ylabel("Frecuencia")
plt.show()
# Asimetria positiva: la mayoria de diamantes son de precio bajo/medio

In [ ]:
# Distribucion de variables categoricas
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.countplot(data=df, x="cut", ax=axes[0],
              order=["Fair", "Good", "Very Good", "Premium", "Ideal"])
axes[0].set_title("Distribucion de Cut")

sns.countplot(data=df, x="color", ax=axes[1],
              order=["J", "I", "H", "G", "F", "E", "D"])
axes[1].set_title("Distribucion de Color")

sns.countplot(data=df, x="clarity", ax=axes[2],
              order=["I1", "SI2", "SI1", "VS2", "VS1", "VVS2", "VVS1", "IF"])
axes[2].set_title("Distribucion de Clarity")

plt.tight_layout()
plt.show()

In [ ]:
# Relacion carat vs price
plt.figure(figsize=(10, 5))
sns.scatterplot(data=df, x="carat", y="price", alpha=0.3)
plt.title("Relacion entre Quilates y Precio")
plt.show()
# Tendencia exponencial: a mayor carat, mayor precio con mayor dispersion

In [ ]:
# Precio segun cada variable categorica
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.boxplot(data=df, x="cut", y="price", ax=axes[0],
            order=["Fair", "Good", "Very Good", "Premium", "Ideal"])
axes[0].set_title("Precio segun Cut")

sns.boxplot(data=df, x="color", y="price", ax=axes[1],
            order=["J", "I", "H", "G", "F", "E", "D"])
axes[1].set_title("Precio segun Color")

sns.boxplot(data=df, x="clarity", y="price", ax=axes[2],
            order=["I1", "SI2", "SI1", "VS2", "VS1", "VVS2", "VVS1", "IF"])
axes[2].set_title("Precio segun Clarity")

plt.tight_layout()
plt.show()
# Nota: cortes Fair tienen mediana de precio mayor que Ideal
# Esto es una variable de confusion: los Fair suelen ser mas pesados (mas carat)

In [ ]:
# Mapa de correlacion
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(numeric_only=True), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Mapa de Correlacion")
plt.tight_layout()
plt.show()
# carat-price: ~0.92 (fuerte predictor)
# x, y, z: multicolinealidad severa entre si (>0.95)

In [ ]:
# =============================================================
# 3. DATA PREPARATION (Parte I)
# =============================================================

In [ ]:
# Remocion de registros con dimensiones fisicas = 0
# Justificacion: es fisicamente imposible, son errores de captura
# Representan <0.04% de la muestra, impacto insignificante
print(f"Antes de limpieza: {len(df)} registros")
df = df[(df["x"] > 0) & (df["y"] > 0) & (df["z"] > 0)]
print(f"Despues de limpieza: {len(df)} registros")

In [ ]:
# Verificar que las variables categoricas estan limpias
for col in ["cut", "color", "clarity"]:
    print(f"{col}: {df[col].unique().tolist()}")
# Todas las categorias son validas y estandar de la industria

In [ ]:
# Separar features (X) y variable objetivo (y)
X = df.drop("price", axis=1)
y = df["price"]
print(f"Features: {X.shape}")
print(f"Objetivo: {y.shape}")

In [ ]:
# Codificacion de variables categoricas con OrdinalEncoder
# Se usa OrdinalEncoder porque cut, color y clarity son ORDINALES
# Tienen un orden intrinseco definido por la industria gemologica
# One-Hot destruiria esa informacion de orden

cut_order = ["Fair", "Good", "Very Good", "Premium", "Ideal"]
color_order = ["J", "I", "H", "G", "F", "E", "D"]
clarity_order = ["I1", "SI2", "SI1", "VS2", "VS1", "VVS2", "VVS1", "IF"]

encoder = OrdinalEncoder(categories=[cut_order, color_order, clarity_order])
X[["cut", "color", "clarity"]] = encoder.fit_transform(X[["cut", "color", "clarity"]])

print("Variables codificadas:")
X[["cut", "color", "clarity"]].head()

In [ ]:
# Estado final del dataset preparado
print(f"Shape final: {X.shape}")
print(f"\nTipos de datos:")
print(X.dtypes)
print(f"\nNulos restantes: {X.isnull().sum().sum()}")
X.head()